# ROCKET: remove kernels and re-transform

Fit a ROCKET transform, then poke inside the fitted object to drop a few kernels and see whether transforming again crashes.

In [1]:
import numpy as np
from aeon.transformations.collection.convolution_based import Rocket
from aeon.datasets import load_unit_test

X_train, y_train = load_unit_test(split="train")
X_test, y_test = load_unit_test(split="test")
X_train.shape, X_test.shape

((20, 1, 24), (22, 1, 24))

In [2]:
# Fit + transform a small ROCKET
n_kernels = 100
rocket = Rocket(n_kernels=n_kernels, random_state=0)
feats = rocket.fit_transform(X_train)
print("features shape:", feats.shape)  # (n_cases, 2 * n_kernels)

features shape: (20, 200)


## Inspect the fitted kernels

`rocket.kernels` is the tuple returned by `_generate_kernels`:
`(weights, lengths, biases, dilations, paddings, num_channel_indices, channel_indices)`.

Note the per-kernel arrays (`lengths`, `biases`, `dilations`, `paddings`, `num_channel_indices`) have length `n_kernels`, but `weights` and `channel_indices` are *flattened/ragged*: each kernel `i` owns `lengths[i] * num_channel_indices[i]` weights and `num_channel_indices[i]` channel indices, laid out back-to-back. So you can't just slice by kernel index.

In [3]:
kernels = rocket.kernels
names = [
    "weights",
    "lengths",
    "biases",
    "dilations",
    "paddings",
    "num_channel_indices",
    "channel_indices",
]
for name, arr in zip(names, kernels):
    print(f"{name:20s} shape={arr.shape} dtype={arr.dtype}")

weights              shape=(876,) dtype=float32
lengths              shape=(100,) dtype=int32
biases               shape=(100,) dtype=float32
dilations            shape=(100,) dtype=int32
paddings             shape=(100,) dtype=int32
num_channel_indices  shape=(100,) dtype=int32
channel_indices      shape=(100,) dtype=int32


## Naive attempt: chop the per-kernel arrays only

Slice off the last few kernels from the per-kernel arrays and leave the ragged `weights` / `channel_indices` untouched.

**Result: this does *not* crash.** `_apply_kernels` iterates over `len(lengths)` kernels and reads `weights` / `channel_indices` via cumulative offsets built from `lengths` and `num_channel_indices`. Because we dropped kernels from the *end*, every kept kernel's offset is unchanged and the leftover tail weights are simply never read — you get correct features for the kept kernels. It would only break if you removed kernels from the *middle/front* (offsets desync → wrong values) or *added* kernels (out-of-bounds read).

In [4]:
import copy

rocket_naive = copy.deepcopy(rocket)
weights, lengths, biases, dilations, paddings, nci, ci = rocket_naive.kernels

drop = 5
keep = n_kernels - drop
rocket_naive.kernels = (
    weights,  # left untouched on purpose
    lengths[:keep],
    biases[:keep],
    dilations[:keep],
    paddings[:keep],
    nci[:keep],
    ci,  # left untouched on purpose
)

try:
    out = rocket_naive.transform(X_test)
    print("no crash, output shape:", out.shape)
except Exception as e:
    print("CRASHED:", type(e).__name__, "->", e)

no crash, output shape: (22, 190)


## Correct way: drop kernels consistently across the ragged layout

To actually remove kernels we must also slice the matching ranges out of `weights` and `channel_indices`. Below we drop the last `drop` kernels properly by recomputing the flattened offsets.

In [5]:
rocket_ok = copy.deepcopy(rocket)
weights, lengths, biases, dilations, paddings, nci, ci = rocket_ok.kernels

drop = 5
keep = n_kernels - drop

# how many weights / channel indices the kept kernels occupy
weights_per_kernel = lengths.astype(np.int64) * nci.astype(np.int64)
n_weights_keep = int(weights_per_kernel[:keep].sum())
n_ci_keep = int(nci[:keep].sum())

rocket_ok.kernels = (
    weights[:n_weights_keep],
    lengths[:keep],
    biases[:keep],
    dilations[:keep],
    paddings[:keep],
    nci[:keep],
    ci[:n_ci_keep],
)
rocket_ok.n_kernels = keep

out = rocket_ok.transform(X_test)
print("output shape:", out.shape, "(expected", (X_test.shape[0], 2 * keep), ")")

output shape: (22, 190) (expected (22, 190) )


In [6]:
# Sanity check: the kept kernels produce the same features as before the drop
full_out = rocket.transform(X_test)
np.allclose(out, full_out[:, : 2 * keep])

True

## `SubsetRocket`: tell ROCKET which features to produce

Each kernel `k` produces two output columns in order `[max_pool, ppv]`, so feature index `f` maps to kernel `f // 2` and `f % 2` selects max (0) vs PPV (1).

`transform_subset(X, features)` figures out the unique kernels those features need, builds a reduced kernel set (slicing the ragged `weights` / `channel_indices` correctly for an *arbitrary* subset, not just a suffix), runs the convolution on only those kernels, then returns the requested columns **in the order you asked** (duplicates allowed). This is faster than computing all features and slicing, since unused kernels are never convolved.

In [7]:
import copy

import numpy as np
from aeon.transformations.collection.convolution_based import Rocket


def _subset_kernels(kernels, keep):
    """Return a new ROCKET kernel tuple keeping only the kernels in `keep`.

    Handles the ragged `weights` / `channel_indices` buffers by slicing each
    kept kernel's block out via cumulative offsets. `keep` may be any index
    array (arbitrary order / subset), not just a suffix.
    """
    weights, lengths, biases, dilations, paddings, nci, ci = kernels
    keep = np.asarray(keep, dtype=np.int64)

    # cumulative start offsets into the flattened buffers
    w_per = lengths.astype(np.int64) * nci.astype(np.int64)
    w_off = np.concatenate([[0], np.cumsum(w_per)])
    ci_off = np.concatenate([[0], np.cumsum(nci.astype(np.int64))])

    if len(keep):
        new_weights = np.concatenate([weights[w_off[k] : w_off[k + 1]] for k in keep])
        new_ci = np.concatenate([ci[ci_off[k] : ci_off[k + 1]] for k in keep])
    else:
        new_weights = weights[:0]
        new_ci = ci[:0]

    return (
        new_weights.astype(np.float32),
        lengths[keep],
        biases[keep],
        dilations[keep],
        paddings[keep],
        nci[keep],
        new_ci.astype(np.int32),
    )


class SubsetRocket(Rocket):
    """ROCKET that can transform only a chosen subset of output features."""

    def transform_subset(self, X, features):
        """Transform `X` producing only the requested `features`.

        Parameters
        ----------
        features : 1D int array-like
            Feature indices in ``range(2 * n_kernels)``. Column ``2*k`` is the
            max-pool of kernel ``k`` and ``2*k+1`` is its PPV. Order is
            preserved and duplicates are allowed.

        Returns
        -------
        np.ndarray of shape (n_cases, len(features))
        """
        features = np.asarray(features, dtype=np.int64)
        needed = np.unique(features // 2)  # sorted unique kernels

        sub = copy.deepcopy(self)
        sub.kernels = _subset_kernels(self.kernels, needed)
        sub.n_kernels = len(needed)

        out = sub.transform(X)  # columns: for each kept kernel, [max, ppv]

        pos = {int(k): i for i, k in enumerate(needed)}
        cols = np.array([2 * pos[int(f // 2)] + int(f % 2) for f in features])
        return out[:, cols]


In [8]:
# Fit a SubsetRocket and verify transform_subset matches the full transform
sr = SubsetRocket(n_kernels=100, random_state=0)
full = sr.fit_transform(X_train)  # (n_cases, 200)
full_test = sr.transform(X_test)

# pick an arbitrary, out-of-order set of features (mix of max and ppv columns)
wanted = [5, 0, 199, 198, 42, 7, 0]  # note: duplicate (0) and reversed order
subset = sr.transform_subset(X_test, wanted)

print("subset shape:", subset.shape, "(expected", (X_test.shape[0], len(wanted)), ")")
print("matches full transform:", np.allclose(subset, full_test[:, wanted]))

subset shape: (22, 7) (expected (22, 7) )
matches full transform: True
